# Imports

In [21]:
import os
import optuna
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from xgboost import XGBClassifier

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, StratifiedKFold

from feature_engine.selection import DropFeatures

In [2]:
os.environ["XGBOOST_VERBOSITY"] = "0"

## Utils

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/X_train.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')

y_train = pd.read_parquet('../data/y_train.parquet')

In [5]:
X_train.head()

,ordinal_encoder__compound,target_encoder__driver,target_encoder__compound,target_encoder__race,standard_scaler__lapnumber,standard_scaler__position,standard_scaler__raceprogress,standard_scaler__year,robust_scaler__position_change,robust_scaler__cumulative_degradation,...,remainder__fuzzy_1_position_position_change,remainder__fuzzy_2_position_position_change,remainder__fuzzy_3_position_position_change,remainder__fuzzy_0_raceprogress_lapnumber,remainder__fuzzy_1_raceprogress_lapnumber,remainder__fuzzy_2_raceprogress_lapnumber,remainder__lap_time_inv,remainder__position_norm,remainder__is_pit_lap,remainder__race_progress_sin
id,,,,,,,,,,,,,,,,,,,,,
0,0.50,0.203815,0.328482,0.154734,1.585901,-0.308849,1.487008,-1.486487,1.666667,1.040769,...,0.947453,0.005655,0.020178,0.000549,0.997580,0.001872,0.012740,0.40,0,0.781831
1,0.50,0.199575,0.327181,0.176114,0.229628,-1.066602,0.033534,1.440545,-1.000000,-5.009333,...,0.074107,0.081854,0.098684,0.015175,0.007697,0.977128,0.013316,0.20,1,0.885456
2,0.50,0.218840,0.326837,0.187526,2.116616,0.638343,1.902200,-1.486487,1.000000,-1.970285,...,0.226475,0.059038,0.614716,0.024077,0.912645,0.063278,0.014095,0.65,0,0.537300
3,0.25,0.200526,0.100888,0.145166,-1.244581,-0.498287,-1.029455,-0.510810,0.000000,0.338641,...,0.088099,0.041344,0.113803,0.950726,0.010153,0.039121,0.010598,0.35,0,0.239316
4,0.50,0.192082,0.327657,0.214557,0.170660,-1.445478,0.092589,-1.486487,1.000000,0.169816,...,0.256086,0.040890,0.083385,0.009589,0.004844,0.985566,0.009270,0.10,1,0.906308


In [6]:
X_test.head()

,ordinal_encoder__compound,target_encoder__driver,target_encoder__compound,target_encoder__race,standard_scaler__lapnumber,standard_scaler__position,standard_scaler__raceprogress,standard_scaler__year,robust_scaler__position_change,robust_scaler__cumulative_degradation,...,remainder__fuzzy_1_position_position_change,remainder__fuzzy_2_position_position_change,remainder__fuzzy_3_position_position_change,remainder__fuzzy_0_raceprogress_lapnumber,remainder__fuzzy_1_raceprogress_lapnumber,remainder__fuzzy_2_raceprogress_lapnumber,remainder__lap_time_inv,remainder__position_norm,remainder__is_pit_lap,remainder__race_progress_sin
id,,,,,,,,,,,,,,,,,,,,,
439140,0.25,0.182308,0.101132,0.133462,-0.124182,-1.066602,0.261317,-0.510810,0.000000,0.396609,...,0.001144,0.000465,0.000894,0.067702,0.028176,0.904122,0.010708,0.20,0,0.954721
439141,0.25,0.356316,0.101132,0.150547,0.052723,-1.634917,0.300590,-0.510810,0.000000,0.470778,...,0.081596,0.033933,0.053686,0.019477,0.011500,0.969023,0.011005,0.05,0,0.963550
439142,0.25,0.121796,0.101132,0.133462,0.052723,0.259466,0.489100,-0.510810,0.000000,0.301036,...,0.087683,0.072629,0.701444,0.038897,0.031448,0.929655,0.010768,0.55,0,0.992709
439143,0.00,0.198847,0.193475,0.253713,-1.008708,1.017219,-1.025511,0.464868,0.333333,0.724449,...,0.023113,0.022859,0.935769,0.984826,0.002887,0.012287,0.010530,0.75,0,0.242362
439144,0.50,0.197127,0.327536,0.114051,1.703838,0.448905,1.518343,0.464868,2.333333,0.003617,...,0.718809,0.040632,0.151701,0.002043,0.991340,0.006618,0.010090,0.60,0,0.766044


## Columns

In [7]:
X_train.columns

Index(['ordinal_encoder__compound', 'target_encoder__driver',
       'target_encoder__compound', 'target_encoder__race',
       'standard_scaler__lapnumber', 'standard_scaler__position',
       'standard_scaler__raceprogress', 'standard_scaler__year',
       'robust_scaler__position_change',
       'robust_scaler__cumulative_degradation', 'robust_scaler__laptime_delta',
       'robust_scaler__laptime_s', 'robust_scaler__stint',
       'robust_scaler__driver_mean_lap', 'robust_scaler__tyrelife',
       'robust_scaler__delta_x_tyre_life', 'robust_scaler__compound_tyre_life',
       'robust_scaler__stint_progress', 'robust_scaler__tyre_life_ratio',
       'robust_scaler__degradation_per_lap',
       'robust_scaler__position_change_cum', 'robust_scaler__laps_since_pit',
       'remainder__pitstop', 'remainder__lapnumber_low',
       'remainder__lapnumber_medium', 'remainder__lapnumber_high',
       'remainder__stint_low', 'remainder__stint_medium',
       'remainder__stint_high', 'remainde

In [8]:
X_test.columns

Index(['ordinal_encoder__compound', 'target_encoder__driver',
       'target_encoder__compound', 'target_encoder__race',
       'standard_scaler__lapnumber', 'standard_scaler__position',
       'standard_scaler__raceprogress', 'standard_scaler__year',
       'robust_scaler__position_change',
       'robust_scaler__cumulative_degradation', 'robust_scaler__laptime_delta',
       'robust_scaler__laptime_s', 'robust_scaler__stint',
       'robust_scaler__driver_mean_lap', 'robust_scaler__tyrelife',
       'robust_scaler__delta_x_tyre_life', 'robust_scaler__compound_tyre_life',
       'robust_scaler__stint_progress', 'robust_scaler__tyre_life_ratio',
       'robust_scaler__degradation_per_lap',
       'robust_scaler__position_change_cum', 'robust_scaler__laps_since_pit',
       'remainder__pitstop', 'remainder__lapnumber_low',
       'remainder__lapnumber_medium', 'remainder__lapnumber_high',
       'remainder__stint_low', 'remainder__stint_medium',
       'remainder__stint_high', 'remainde

## Data Dimension

In [9]:
X_train.shape

(439140, 60)

In [10]:
X_test.shape

(188165, 60)

## Check NA

In [11]:
X_train.isna().mean()

ordinal_encoder__compound                             0.0
target_encoder__driver                                0.0
target_encoder__compound                              0.0
target_encoder__race                                  0.0
standard_scaler__lapnumber                            0.0
standard_scaler__position                             0.0
standard_scaler__raceprogress                         0.0
standard_scaler__year                                 0.0
robust_scaler__position_change                        0.0
robust_scaler__cumulative_degradation                 0.0
robust_scaler__laptime_delta                          0.0
robust_scaler__laptime_s                              0.0
robust_scaler__stint                                  0.0
robust_scaler__driver_mean_lap                        0.0
robust_scaler__tyrelife                               0.0
robust_scaler__delta_x_tyre_life                      0.0
robust_scaler__compound_tyre_life                     0.0
robust_scaler_

In [12]:
X_test.isna().mean()

ordinal_encoder__compound                             0.0
target_encoder__driver                                0.0
target_encoder__compound                              0.0
target_encoder__race                                  0.0
standard_scaler__lapnumber                            0.0
standard_scaler__position                             0.0
standard_scaler__raceprogress                         0.0
standard_scaler__year                                 0.0
robust_scaler__position_change                        0.0
robust_scaler__cumulative_degradation                 0.0
robust_scaler__laptime_delta                          0.0
robust_scaler__laptime_s                              0.0
robust_scaler__stint                                  0.0
robust_scaler__driver_mean_lap                        0.0
robust_scaler__tyrelife                               0.0
robust_scaler__delta_x_tyre_life                      0.0
robust_scaler__compound_tyre_life                     0.0
robust_scaler_

# Machine Learning

## Base

In [13]:
cv_results = cross_val_score(
    estimator=XGBClassifier(random_state=42, verbosity=0), 
    X=X_train, 
    y=y_train.PitNextLap, 
    scoring='roc_auc', 
    cv=StratifiedKFold(random_state=42, shuffle=True), 
    n_jobs=-1
)

In [14]:
cv_results.mean()

np.float64(0.9452400416659223)

In [15]:
model = XGBClassifier(random_state=42, verbosity=0).fit(X_train, y_train.PitNextLap)

## Feature Selection

In [26]:
perm_result = permutation_importance(
    estimator=model, 
    X=X_train, 
    y=y_train.PitNextLap, 
    n_jobs=-1, 
    scoring='roc_auc'
)


importance_df = pd.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

In [27]:
importance_df

,feature,importance_mean,importance_std
12,robust_scaler__stint,0.082732,0.000323
7,standard_scaler__year,0.066972,0.000439
18,robust_scaler__tyre_life_ratio,0.025760,0.000117
14,robust_scaler__tyrelife,0.023285,0.000123
3,target_encoder__race,0.011125,0.000047
10,robust_scaler__laptime_delta,0.007145,0.000064
15,robust_scaler__delta_x_tyre_life,0.004168,0.000069
8,robust_scaler__position_change,0.003230,0.000034
9,robust_scaler__cumulative_degradation,0.003154,0.000063
36,remainder__raceprogress_medium,0.002551,0.000022


In [18]:
importance_df.query("importance_mean <= 0").feature.tolist()

['remainder__stint_low',
 'remainder__lapnumber_high',
 'remainder__lapnumber_low',
 'remainder__position_low',
 'remainder__stint_high',
 'remainder__position_high',
 'remainder__position_norm',
 'remainder__is_pit_lap']

In [22]:
drop_feat = DropFeatures(features_to_drop=importance_df.query("importance_mean <= 0").feature.tolist())

## Fine Tuning

In [23]:
def objective(trial, X, y):

    scale_pos_weight = (y.PitNextLap == 0).sum() / (y.PitNextLap == 1).sum()
    
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "verbosity": 0,
        "scale_pos_weight": scale_pos_weight,
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]
        
        model = Pipeline([('drop_feat', drop_feat), ('xgb', XGBClassifier(**params))])
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]
        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=50, n_jobs=-1, show_progress_bar=True)

[I 2026-05-05 14:16:55,046] A new study created in memory with name: no-name-67a65bf9-df40-4c05-a9e3-e9333bf68800


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-05 14:18:11,364] Trial 8 finished with value: 0.9323216108047673 and parameters: {'max_depth': 4, 'learning_rate': 0.032490127141286615, 'n_estimators': 124, 'subsample': 0.7518158280578091, 'colsample_bytree': 0.8777085073261768, 'gamma': 4.8285610052348416, 'reg_alpha': 0.027942409699874375, 'reg_lambda': 4.2012410421555716e-08}. Best is trial 8 with value: 0.9323216108047673.
[I 2026-05-05 14:20:54,172] Trial 4 finished with value: 0.9346356765216512 and parameters: {'max_depth': 8, 'learning_rate': 0.0010677570335650578, 'n_estimators': 205, 'subsample': 0.8491472396768136, 'colsample_bytree': 0.5409403469710052, 'gamma': 0.6761497156933138, 'reg_alpha': 9.865973324902028e-06, 'reg_lambda': 0.2874479211800762}. Best is trial 4 with value: 0.9346356765216512.
[I 2026-05-05 14:23:52,842] Trial 1 finished with value: 0.9347044163554827 and parameters: {'max_depth': 4, 'learning_rate': 0.006723999390717626, 'n_estimators': 815, 'subsample': 0.8518440170494503, 'colsample_byt

In [24]:
scale_pos_weight = (y_train.PitNextLap == 0).sum() / (y_train.PitNextLap == 1).sum()

model_tuned = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    verbosity=0,
    scale_pos_weight=scale_pos_weight,
    **study.best_params
)

pipe_tuned = Pipeline([('drop_feat', drop_feat), ('xgb', model_tuned)])
pipe_tuned.fit(X_train, y_train.PitNextLap)

Pipeline(steps=[('drop_feat',
                 DropFeatures(features_to_drop=['remainder__stint_low',
                                                'remainder__lapnumber_high',
                                                'remainder__lapnumber_low',
                                                'remainder__position_low',
                                                'remainder__stint_high',
                                                'remainder__position_high',
                                                'remainder__position_norm',
                                                'remainder__is_pit_lap'])),
                ('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_byleve...
                               gamma=1.062979397487378, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.02351208051350585, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=10,
                               max_leaves=None, min_child_weight=None,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=910,
                               n_jobs=None, num_parallel_tree=None, ...))])

# Deploy

In [25]:
dump_pickle(pipe_tuned, '../models/model_xgboost.pkl')